In [1]:
from sklearn.cluster import KMeans
import pandas as pd
import numpy as np
from plotnine import ggplot, aes, geom_point, facet_grid, labs, scale_y_continuous, labeller, as_labeller, scale_x_continuous
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

In [2]:
# read in data
url = "https://huggingface.co/boettiger-lab/rl4eco/resolve/main/rl4greencrab/data/rl_policies/tqc_clean.csv"
data = pd.read_csv(url)

In [3]:
# get rid of t = 0:
data = data[data["t"] > 0]

In [4]:
# remove anomolous biomass data
data = data[(data['biomass'] != -1) & (data['biomass'] <= -0.46)]
data = data.iloc[5:]

In [5]:
# subset to relevant columns
subset = data[['months', 'act0', 'act1', 'CPUE', 'biomass']]

In [6]:
# loop through months and actions to cluster data
months = subset['months'].unique()
actions = ['act0', 'act1']
all_centroids = []
all_labeled = []

for month in months:
    data_month = subset[subset['months'] == month].drop(columns=['months'])
    
    for action in actions:
        other_action = 'act1' if action == 'act0' else 'act0'
        X = data_month.drop(columns=[other_action])
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        if month in [4, 5, 6, 7, 8] and action == 'act0':
            k = 1
        elif month in [9, 10] and action == 'act0':
            k = 2
        else:
            k = 3
        
        kmeans = KMeans(n_clusters=k, n_init=30, random_state=42).fit(X_scaled)
        
        # build labeled chunk
        chunk = X.copy().reset_index(drop=True)
        chunk['month'] = month
        chunk['action'] = action
        chunk['cluster'] = kmeans.labels_
        all_labeled.append(chunk)
        
        # store centroids
        centroids_original = scaler.inverse_transform(kmeans.cluster_centers_)
        centroids_df = pd.DataFrame(centroids_original, columns=X.columns)
        centroids_df['month'] = month
        centroids_df['action'] = action
        centroids_df['cluster'] = range(k)
        all_centroids.append(centroids_df)



In [7]:
labeled = pd.concat(all_labeled, ignore_index=True)
centroids = pd.concat(all_centroids, ignore_index=True)

In [8]:
labeled.head()

,act0,CPUE,biomass,month,action,cluster,act1
0,-0.964028,-0.981809,-0.879119,8,act0,0,NaN
1,-0.913234,-0.981059,-0.695920,8,act0,0,NaN
2,-0.918988,-0.975811,-0.739296,8,act0,0,NaN
3,-0.915355,-0.987716,-0.526255,8,act0,0,NaN
4,-0.916157,-0.989578,-0.536008,8,act0,0,NaN


In [9]:
centroids.head()

,act0,CPUE,biomass,month,action,cluster,act1
0,-0.924725,-0.987401,-0.655309,8,act0,0,NaN
1,NaN,-0.985756,-0.778625,8,act1,0,-0.289208
2,NaN,-0.990544,-0.603308,8,act1,1,0.326378
3,NaN,-0.966580,-0.836234,8,act1,2,-0.153271
4,-0.963139,-0.973723,-0.782643,9,act0,0,NaN


In [10]:
# save labeled data and centroids
labeled.to_csv("../data/cluster/labeled.csv", index=False)
centroids.to_csv("../data/cluster/centroids.csv", index=False)
